In [ ]:
import numpy as np
from itertools import product
import matplotlib.pyplot as plt
from matplotlib import font_manager
from glob import glob
import os
import json
import seaborn as sns
import pandas as pd
from collections import Counter

font_path = 'Futura Book.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path, size='large')
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams.update({'font.size': 16})

In [ ]:
stacked = []
for path in glob("./data/*/amplitudes/*json"):
    splitpath = path.split("/")
    basis = splitpath[2]
    name = splitpath[-1].split('_')[2]
    with open(path,'r') as f:
        metrics = json.load(f)
    
    
    gendf = pd.DataFrame.from_dict(metrics['amplitude_generation_times'],orient='index').rename(columns={0:'amplitude_generation_times'})
    metricdf = pd.DataFrame.from_dict(metrics['metrics'],orient='index').rename(columns={0:'metrics'})
    metricdf.loc['t2',metricdf.columns] = 0
    gendf = gendf.rename(index={'Zeroes':'t2_zeroes', 'Random':'t2_rand', 'MP2*':'t2_MP2', "ML":'t2_ML',"Exact":'t2', 'ML_exact':'t2_ML_exact'})

    joined = pd.concat([metricdf,gendf],axis=1)
    joined['molecule'] = metrics['molecule']
    joined['basis'] = metrics['basis']
    
    # gendf.index = metricdf.index
    stacked.append(joined)

stackeddf = pd.concat(stacked).reset_index().rename(columns={'index':'Amplitude'})
stackeddf = stackeddf[stackeddf['Amplitude']!='t2']
stackeddf.sort_values(by=['basis','molecule','Amplitude'],inplace=True)
# stackeddf['Amplitude'] = stackeddf['Amplitude'].map({'t2_zeroes':r't$_{2}^{\mathrm{Zeroes}}$', 't2_rand':r't$_{2}^{\mathrm{Random}}$', 't2_MP2':r't$_{2}^{\mathrm{MP2}}$', 't2_ML':r't$_{2}^{\mathrm{ML}}$', 't2_ML_exact':r't$_{2}^{\mathrm{ML(exact)}}$', 't2':r't$_{2}$'})
stackeddf['Injection'] = stackeddf['Amplitude'].map({'t2_zeroes':'zeroes', 't2_rand':'random', 't2_MP2':'MP2', 't2_ML':'ML', 't2_ML_exact':'ML_exact', 't2':'CCSD'})
stackeddf.rename(columns={'molecule':'Name','basis':'Basis'},inplace=True)

In [ ]:
data_amp = pd.concat([stackeddf.loc[stackeddf['Amplitude']==i,'MAPE'].describe().to_frame().rename(columns={'MAPE':i}) for i in stackeddf['Amplitude'].drop_duplicates().values],axis=1)

In [ ]:
data_amp.style.format('{:.2e}')

In [ ]:
UofT_palette = [ "#1E3765",
                 "#007FA3", 
                 "#6D247A", 
                 "#DC4633",
                 "#6FC7EA",
                 "#00A189",
                 "#AB1368",
                 "#0D534D",
                 "#F1C500",
                 "#8DBF2E"
               ]

palette = sns.color_palette(UofT_palette)

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(stackeddf,x='Injection',y='MAPE',hue='Basis',hue_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],palette=palette[0:3],order=['zeroes','random','MP2','ML','ML_exact'])
plt.yscale("log")
plt.ylim(10**-12,10**12)
plt.xlabel("")
plt.legend(bbox_to_anchor=(1.3, 0.9))
plt.tight_layout()
plt.savefig("./GMJ_figures/AmpMapeBoxplot.png",dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(stackeddf,x='Injection',y='MSE',hue='Basis',hue_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],palette=palette[0:3],order=['zeroes','random','MP2','ML','ML_exact'])
plt.yscale("log")
# plt.ylim(10**-12,10**12)
plt.xlabel("")
plt.legend(bbox_to_anchor=(1.3, 0.9))
plt.tight_layout()
plt.savefig("./GMJ_figures/AmpMSEBoxplot.png",dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.boxplot(stackeddf,x='Injection',y='MAE',hue='Basis',hue_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],palette=palette[0:3],order=['zeroes','random','MP2','ML','ML_exact'])
plt.yscale("log")
plt.ylim(10**-12,10**12)
plt.xlabel("")
plt.legend(bbox_to_anchor=(1.3, 0.9))
plt.tight_layout()
plt.savefig("./GMJ_figures/AmpMAEBoxplot.png",dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
for inj in ['zeroes','random','MP2','ML','ML_exact']:
    print(inj,'\n',stackeddf.loc[(stackeddf['Injection'] == inj), 'MAE'].describe().loc[['mean', 'min', 'max', 'std']].map('{:.2e}'.format))

In [ ]:
stackeddf.idxmax

In [ ]:
stackeddf.loc[(stackeddf['Basis']=="STO-3G"),'MAPE'].idxmax()

In [ ]:
for amp in ['zeroes','random','MP2','ML','ML_exact']:
    stats = stackeddf.loc[(stackeddf['Injection']==amp),'MAPE'].describe()
    print(amp)
    print(stats.loc[['mean','min','max','std']].map('{:.2e}'.format))
    print()

In [ ]:
moleculeinloop = []
for amp, basis in product(['zeroes','random','MP2','ML','ML_exact'],['STO-3G','cc-pVDZ','aug-cc-pVDZ']):
    found = stackeddf.loc[(stackeddf['Injection']==amp)&(stackeddf['Basis']==basis),'MAE']
    stats = found.describe()
    print(amp,basis)
    print(stackeddf.loc[found.idxmax(),'Name'],found.idxmax(),f"{found.max():.2e}")
    # print(stats.loc[['mean','std','min','max']].round(2))
    moleculeinloop.append(stackeddf.loc[found.idxmax(),'Name'])
    print()

freqmolmax = Counter(moleculeinloop)

In [ ]:
freqmolmax

In [ ]:
stackeddf.loc[stackeddf['MAPE'].idxmin()]

In [ ]:
g = sns.catplot(stackeddf,x='Name',y='MAPE',hue='Injection',col='Basis',kind='bar',col_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],palette=palette[0:5],hue_order=['zeroes','random','MP2','ML','ML_exact'])
for ax in g.axes.flat:
    ax.set_yscale("log")
    ax.set_xlabel("Molecule")
    for label in ax.get_xticklabels():
        # print(label)
        label.set_rotation(90)
        label.set_horizontalalignment("center")
        label.set_verticalalignment("top")    
g.figure.set_size_inches(15, 8)
g.set_titles(col_template="{col_name}", row_template="{row_name}") 
handles, labels = g.axes.flat[-1].get_legend_handles_labels()
g.legend.remove()
g.figure.legend(handles, labels, bbox_to_anchor=(1.00, 0.5), loc='center left')
plt.tight_layout()
plt.savefig("./GMJ_figures/AmpMapeMolecules.png",dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
g = sns.catplot(stackeddf,x='Name',y='MSE',hue='Injection',col='Basis',kind='bar',col_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],palette=palette[0:5],hue_order=['zeroes','random','MP2','ML','ML_exact'])
for ax in g.axes.flat:
    ax.set_yscale("log")
    ax.set_xlabel("Molecule")
    for label in ax.get_xticklabels():
        # print(label)
        label.set_rotation(90)
        label.set_horizontalalignment("center")
        label.set_verticalalignment("top")    
g.figure.set_size_inches(15, 8)
g.set_titles(col_template="{col_name}", row_template="{row_name}") 
handles, labels = g.axes.flat[-1].get_legend_handles_labels()
g.legend.remove()
g.figure.legend(handles, labels, bbox_to_anchor=(1.00, 0.5), loc='center left')
plt.tight_layout()
plt.savefig("./GMJ_figures/AmpMSEMolecules.png",dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
g = sns.catplot(stackeddf,x='Name',y='MAE',hue='Injection',col='Basis',kind='bar',col_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],palette=palette[0:5],hue_order=['zeroes','random','MP2','ML','ML_exact'])
for ax in g.axes.flat:
    ax.set_yscale("log")
    ax.set_xlabel("Molecule")
    for label in ax.get_xticklabels():
        # print(label)
        label.set_rotation(90)
        label.set_horizontalalignment("center")
        label.set_verticalalignment("top")    
g.figure.set_size_inches(15, 8)
g.set_titles(col_template="{col_name}", row_template="{row_name}") 
handles, labels = g.axes.flat[-1].get_legend_handles_labels()
g.legend.remove()
g.figure.legend(handles, labels, bbox_to_anchor=(1.00, 0.5), loc='center left')
plt.tight_layout()
plt.savefig("./GMJ_figures/AmpMAEMolecules.png",dpi=300,bbox_inches='tight')
plt.show()

# Compare with the Bitstring data

In [ ]:
dim_df = pd.read_excel("Dimensions.xlsx",index_col=0)

In [ ]:
dim_df['PostselectedPercent'] = (dim_df['Postselected'] / dim_df['Shots'])*1e2

In [ ]:
n_atoms_dict = {
    'water': 3,
    'methane': 5,
    'ammonia': 4,
    'ethane': 8,
    'methanol': 6,
    'ethylene': 6,
    'formaldehyde': 4,
    "prop-2-en-1-ol":10,
    "but-1-yne":10,
    "fluoroform":5,
    "buta-1,3-diene":10,
    "(Z)-1-fluoroprop-1-ene":9
}

dim_df['n_atoms'] = dim_df['Name'].map(n_atoms_dict)

In [ ]:
merged_df = stackeddf.merge(dim_df,on=['Name', 'Basis', 'Injection'],how='left')

activespacedf = pd.read_csv("../../../DDLUCJ_active_spaces_unfrozen.csv",delimiter=';')
activespacedf.rename(columns={"molecule":'Name'},inplace=True)

merged_df = merged_df.merge(activespacedf, on=['Name'],how='left')

In [ ]:

g = sns.catplot(merged_df.sort_values(by=["Basis","n_atoms","L"]),x='Injection',y='PostselectedPercent',hue='L',col='Basis',kind='box',col_order=['STO-3G','cc-pVDZ','aug-cc-pVDZ'],palette=palette,row_order=['zeroes','random','MP2','ML','ML_exact'])
